# JAX / Flax NNX baseline CNN: bars and spiral-arm tightness

This is the **CNN side of the CNN vs ViT comparison**. Everything except the model comes from
shared code in `gz2/`, so `04_vit.ipynb` differs from this notebook only in the model class:

| Shared piece | Where |
|---|---|
| Labels and the 70/15/15 split | `data/processed/gz2_bar_winding_v1.csv` (made in `02_cnn_baseline.ipynb`) |
| Crop, resize, flips/rotations | `gz2/pipeline.py` |
| Masked two-head loss, AdamW + warmup/cosine, early stopping | `gz2/jax_train.py` |
| Metrics, size-bin breakdown, paired bootstrap | `gz2/evaluate.py` |

The architecture mirrors the Keras baseline: four conv blocks, global average pooling, and two
heads. **bar** is a logit trained with binary cross-entropy. **winding** is 3 logits, softmaxed
into (tight, medium, loose) and trained with MSE. Loss = `1.0 · bar + 0.5 · winding`.

Each seed's test predictions are saved to `output/predictions/jax_cnn_seed{N}.csv` for the
comparison at the end of `04_vit.ipynb`.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import pandas as pd
from flax import nnx

from gz2.evaluate import breakdown, load_context, load_predictions, metrics, save_predictions
from gz2.jax_train import count_params, fit, predict
from gz2.pipeline import load_split, make_dataset

MODEL_NAME = "jax_cnn"
SEEDS = [0, 1, 2]           # several seeds, so a CNN-vs-ViT gap can be told apart from run-to-run noise
IMG_SIZE = 128
BATCH_SIZE = 64
EPOCHS = 15
LR = 1e-3                   # tune on validation; the ViT gets its own tuned LR
LOSS_WEIGHTS = (1.0, 0.5)   # (bar, winding)
SMOKE_TEST = False          # True = tiny subset, 1 epoch, 1 seed: checks the pipeline end to end

print("JAX", jax.__version__, "| devices:", jax.devices())

## Data

The split is read from disk rather than re-made, so the CNN and ViT see exactly the same
galaxies.

In [ ]:
split = load_split()
if SMOKE_TEST:
    split = split.groupby("split").sample(n=300, random_state=0)
    EPOCHS, SEEDS = 1, SEEDS[:1]

frames = {name: split[split["split"] == name] for name in ("train", "val", "test")}
val_ds = make_dataset(frames["val"], training=False, img_size=IMG_SIZE, batch_size=BATCH_SIZE)
test_ds = make_dataset(frames["test"], training=False, img_size=IMG_SIZE, batch_size=BATCH_SIZE)

pd.DataFrame({
    name: {"galaxies": len(f), "bar labels": int(f["bar_weight"].sum()), "winding labels": int(f["winding_weight"].sum())}
    for name, f in frames.items()
}).T

## Model

Each block is conv 3×3 → batch norm → ReLU → 2×2 max-pool, so a 128 px input ends as an 8×8×256
feature map. Images arrive in [0, 255] and are scaled inside the model, the same as the ViT.

In [ ]:
class ConvBlock(nnx.Module):
    def __init__(self, in_features: int, out_features: int, *, rngs: nnx.Rngs):
        self.conv = nnx.Conv(in_features, out_features, kernel_size=(3, 3), padding="SAME", use_bias=False, rngs=rngs)
        self.norm = nnx.BatchNorm(out_features, rngs=rngs)

    def __call__(self, x: jax.Array) -> jax.Array:
        x = nnx.relu(self.norm(self.conv(x)))
        return nnx.max_pool(x, window_shape=(2, 2), strides=(2, 2))


class GalaxyCNN(nnx.Module):
    def __init__(
        self,
        widths: tuple[int, ...] = (32, 64, 128, 256),
        hidden: int = 128,
        dropout: float = 0.3,
        *,
        rngs: nnx.Rngs,
    ):
        channels = (3, *widths)
        self.blocks = nnx.List([ConvBlock(c_in, c_out, rngs=rngs) for c_in, c_out in zip(channels[:-1], channels[1:])])
        self.dropout = nnx.Dropout(dropout, rngs=rngs)
        self.hidden = nnx.Linear(widths[-1], hidden, rngs=rngs)
        self.bar_head = nnx.Linear(hidden, 1, rngs=rngs)
        self.winding_head = nnx.Linear(hidden, 3, rngs=rngs)

    def __call__(self, images: jax.Array) -> tuple[jax.Array, jax.Array]:
        x = images / 255.0
        for block in self.blocks:
            x = block(x)
        x = x.mean(axis=(1, 2))                     # global average pooling
        x = nnx.relu(self.hidden(self.dropout(x)))
        return self.bar_head(x)[:, 0], self.winding_head(x)


model = GalaxyCNN(rngs=nnx.Rngs(0))
bar_logit, winding_logits = model(jnp.zeros((2, IMG_SIZE, IMG_SIZE, 3)))
print(f"{count_params(model):,} parameters | outputs: bar {bar_logit.shape}, winding {winding_logits.shape}")
nnx.display(model)

## Train one model per seed

Each seed changes the weight initialisation, dropout and shuffling. The best epoch (lowest
validation loss) is kept, and its test predictions are saved.

In [ ]:
histories, test_metrics = {}, {}
for seed in SEEDS:
    print(f"\n=== seed {seed} ===")
    train_ds = make_dataset(frames["train"], training=True, img_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=seed)
    model = GalaxyCNN(rngs=nnx.Rngs(params=seed, dropout=seed))
    best_model, histories[seed] = fit(model, train_ds, val_ds, epochs=EPOCHS, lr=LR, loss_weights=LOSS_WEIGHTS)

    preds = predict(best_model, test_ds)
    test_metrics[seed] = metrics(preds, LOSS_WEIGHTS)
    print(f"saved -> {save_predictions(preds, MODEL_NAME, seed)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
panels = [("val_loss", "Val total loss"), ("val_bar_acc", "Val bar accuracy"), ("val_winding_rmse", "Val winding RMSE")]
for ax, (val_key, title) in zip(axes, panels):
    for seed, hist in histories.items():
        ax.plot(hist["epoch"], hist[val_key], linewidth=2, marker="o", markersize=4, label=f"seed {seed}")
    ax.set_title(title, loc="left")
    ax.set_xlabel("Epoch")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e6e5e0")
axes[0].legend(frameon=False)
fig.tight_layout()
plt.show()

## Test results

Mean ± standard deviation over seeds. Compare these with the trivial baselines from the EDA
(bar accuracy 0.600, winding RMSE 0.203) and the proposal targets (0.90, 0.10).

In [ ]:
per_seed = pd.DataFrame(test_metrics).T[["bar_acc", "bar_auc", "winding_rmse", "winding_mae"]]
summary = per_seed.agg(["mean", "std"])
pd.concat([per_seed.rename(index=lambda s: f"seed {s}"), summary]).style.format("{:.4f}")

### Where does the CNN do well or badly?

The same breakdown runs on the ViT, so the two tables line up row for row. Shown for the first
seed.

In [ ]:
context = load_context()
breakdown(load_predictions(MODEL_NAME)[0], context).style.format(
    {"bar_acc": "{:.3f}", "bar_auc": "{:.3f}", "winding_rmse": "{:.4f}", "winding_mae": "{:.4f}",
     "n_bar": "{:,.0f}", "n_winding": "{:,.0f}"})